# 16｜序列模型_RNN與時間序列預測

把連續時間序列切成視窗，嚴格使用過去預測下一步。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

rng = np.random.default_rng(SEED)


## 課堂小實驗

In [ ]:
from sklearn.linear_model import LinearRegression
t = np.arange(240)
series = np.sin(t / 12) + 0.35 * np.sin(t / 3.5) + rng.normal(0, 0.08, len(t))
window = 12
X = np.stack([series[i:i+window] for i in range(len(series)-window)])
y = series[window:]
split = 170
model = LinearRegression().fit(X[:split], y[:split])
pred = model.predict(X[split:])
persistence = X[split:, -1]
rmse = lambda a,b: float(np.sqrt(np.mean((a-b)**2)))
pd.Series({'window_model_RMSE': rmse(y[split:], pred), 'persistence_RMSE': rmse(y[split:], persistence)})

## 執行結果圖

In [ ]:
idx = np.arange(split+window, len(series))
plt.plot(idx, y[split:], label='actual')
plt.plot(idx, pred, label='window model')
plt.plot(idx, persistence, alpha=.6, label='persistence')
plt.xlabel('time'); plt.ylabel('value'); plt.title('Forecast uses only the previous 12 values')
plt.legend()
savefig('16_sequence_demo')
report('sequence_forecast', {'window_RMSE': rmse(y[split:], pred), 'persistence_RMSE': rmse(y[split:], persistence)})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。